# Phase 4 â€” Synthetic Finnish learner-error generation

This notebook is the presentation and audit layer for the reusable generator in `app.services.error_generation`. Synthetic errors provide controlled labels and exact corrections for the later baseline, but they are a proxy for learner languageâ€”not evidence that every generated construction occurs naturally in learner writing. No model or final data split is created here.

In [1]:
from pathlib import Path
import json
import sys
from collections import Counter

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'app').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.services.error_generation import (
    ERROR_TYPES,
    run_default_error_generation,
    validate_synthetic_record,
)

PROJECT_ROOT

WindowsPath('C:/Users/leanh/OneDrive/Desktop/code/finnish_learning_assistant')

## Chosen taxonomy and linguistic safeguards

Version 1 implements only three categories that can use corpus-attested Finnish forms.

| Label | Eligibility and transformation | Main exclusions |
|---|---|---|
| `CASE_ERROR` | An `ADJ` attached as `amod` agrees with a nominal head; replace it with the same lemma, number and degree but a different observed case. TDT's explicit and FTB's unmarked positive-degree conventions are both supported. | MWT/empty nodes, comparative/superlative/ordinal or unsafe extra morphology, no agreeing nominal head, no attested alternative. |
| `VERB_CONJUGATION` | A finite active indicative verb has exactly one overt matching nominative personal-pronoun subject; replace it with the same lemma and retained features but a different observed person/number. | Ambiguous/missing subject, non-finite or non-indicative form, unsafe morphology, no attested alternative. |
| `AGREEMENT` | An agreeing attributive `ADJ` is replaced by the same lemma, case and degree in the other observed number. | The same structural and morphology exclusions as `CASE_ERROR`. |

The generator edits the verified token character span in `original_text`; it never rebuilds the sentence from tokens. This preserves unrelated Unicode, casing, punctuation, and source whitespace.

## Execute the reusable pipeline

This call regenerates both Phase 4 artifacts from the Phase 3 JSONL. Production logic is imported rather than copied into notebook cells.

In [2]:
run_summary = run_default_error_generation(PROJECT_ROOT)
run_summary

{'schema_version': '1.0',
 'generator_version': 'v1',
 'input': {'processed_path': 'data/processed/finnish_sentences_v1.jsonl',
  'processed_sha256': '1d2f14e1769e26adae58df808d28fc90231fbffedfa2db46528579e1a18b97f5',
  'preprocessing_manifest_path': 'data/processed/preprocessing_manifest_v1.json',
  'preprocessing_manifest_sha256': '68bfc65d16738adeb183c2f62af46a301e0885cc620081e5b22b930a929edbc3',
  'preprocessing_schema_version': '1.0',
  'source_sentences': 33859},
 'configuration': {'random_seed': 42,
  'maximum_examples_per_class': 1000,
  'minimum_examples_per_class': 500,
  'maximum_variants_per_leakage_group': 2,
  'maximum_examples_per_lemma_per_class': 50,
  'minimum_sentence_words': 3,
  'maximum_sentence_words': 30,
  'manual_review_examples_per_class': 20},
 'generation_rules': [{'generation_rule': 'adjective_case_mismatch_v1',
   'error_type': 'CASE_ERROR',
   'description': 'Replace an agreeing adjectival modifier with a corpus-attested form of the same lemma and number

In [3]:
errors_path = PROJECT_ROOT / 'data/errors/synthetic_errors_v1.jsonl'
manifest_path = PROJECT_ROOT / 'data/errors/error_generation_manifest_v1.json'
phase3_path = PROJECT_ROOT / 'data/processed/finnish_sentences_v1.jsonl'
records = [json.loads(line) for line in errors_path.read_text(encoding='utf-8').splitlines()]
sources = {row['source_id']: row for row in (json.loads(line) for line in phase3_path.read_text(encoding='utf-8').splitlines())}
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
len(records), manifest['output']['counts_by_class']

(3000, {'AGREEMENT': 1000, 'CASE_ERROR': 1000, 'VERB_CONJUGATION': 1000})

## Correct â†’ synthetic incorrect examples

Every row below has one intended edit, an exact correction, and an auditable rule name.

In [4]:
example_rows = []
for label in ERROR_TYPES:
    selected = [row for row in records if row['error_type'] == label][:3]
    example_rows.extend({
        'error_type': row['error_type'],
        'correct': row['correct_sentence'],
        'incorrect': row['incorrect_sentence'],
        'correction': row['correction'],
        'rule': row['generation_rule'],
    } for row in selected)
pd.DataFrame(example_rows)

,error_type,correct,incorrect,correction,rule
0,CASE_ERROR,Joulukuussa 2003 valittiin avoimen tarjouskilp...,Joulukuussa 2003 valittiin avointa tarjouskilp...,avoimen,adjective_case_mismatch_v1
1,CASE_ERROR,Keskivero (ruots. mediumränta) oli Ruotsin val...,Keskivero (ruots. mediumränta) oli Ruotsin val...,ylimääräinen,adjective_case_mismatch_v1
2,CASE_ERROR,– Kalliin lainan ottaminen ja halvan lainan my...,– Kalliin lainan ottaminen ja halpa lainan myö...,halvan,adjective_case_mismatch_v1
3,VERB_CONJUGATION,Sinäkös se otit sen viimesen palan ?,Sinäkös se otti sen viimesen palan ?,otit,finite_verb_person_number_mismatch_v1
4,VERB_CONJUGATION,Mä tein sen tahallaan,Mä teki sen tahallaan,tein,finite_verb_person_number_mismatch_v1
5,VERB_CONJUGATION,Sinä puhut kuin Roope Ankka .,Sinä puhuu kuin Roope Ankka .,puhut,finite_verb_person_number_mismatch_v1
6,AGREEMENT,Suomalaiset mies- ja naisopiskelijat polttavat...,Suomalainen mies- ja naisopiskelijat polttavat...,Suomalaiset,adjective_number_agreement_mismatch_v1
7,AGREEMENT,Sopivilla virityksillä ja soveltavalla askarte...,Sopivilla virityksillä ja soveltavalla askarte...,hyvä,adjective_number_agreement_mismatch_v1
8,AGREEMENT,"– Parasta, mitä voimme tehdä estääksemme vasta...","– Parasta, mitä voimme tehdä estääksemme vasta...",vastaavat,adjective_number_agreement_mismatch_v1


## Automatic validation and rejection evidence

Validation checks provenance, leakage identity, rule/label compatibility, exact inverse correction, one-span isolation, attested-form evidence, and the intended morphology change. Invalid candidates are rejected rather than repaired. The manifest also separates eligibility exclusions from sampling/deduplication rejections.

In [5]:
for row in records:
    validate_synthetic_record(row, sources)

pd.DataFrame({
    'quality_check': manifest['quality_checks'].keys(),
    'passed': manifest['quality_checks'].values(),
})

,quality_check,passed
0,phase3_input_hash_verified,True
1,all_source_ids_exist,True
2,all_leakage_groups_match_source,True
3,all_records_validated,True
4,all_synthetic_ids_unique,True
5,all_duplicate_keys_unique,True
6,classes_balanced,True
7,leakage_group_cap_respected,True
8,no_project_split_created,True


In [6]:
pd.DataFrame([
    {'stage': 'eligibility', 'reason': key, 'count': value}
    for key, value in manifest['source_exclusion_counts'].items()
] + [
    {'stage': 'selection', 'reason': key, 'count': value}
    for key, value in manifest['candidate_rejections'].items()
]).sort_values(['stage', 'count'], ascending=[True, False])

,stage,reason,count
0,eligibility,no_eligible_rule,22614
2,eligibility,sentence_length_outside_limits,2198
1,eligibility,sentence_has_mwt_or_empty_node,756
5,selection,class_cap,9548
4,selection,extra_candidates_within_group_and_class_removed,2022
7,selection,lemma_cap,143
3,selection,exact_duplicate_candidates_removed,55
6,selection,leakage_group_cap,5


## Class balance, duplicates, provenance, and shortcut risks

The selection favors a modest balanced dataset. At most one example of a class and two variants total may come from a leakage group; a lemma/class cap reduces repetitive paradigms. Source-treebank and sentence-length distributions are also shown because they can become synthetic shortcuts. Phase 5 must stratify by error type plus source dataset where group constraints allow, normalize formatting-only whitespace for features, and report shortcut diagnostics. The final project split remains Phase 5 work.

In [7]:
distribution = pd.Series(Counter(row['error_type'] for row in records), name='examples').sort_index()
display(distribution.to_frame())
display(pd.Series(manifest['duplicate_statistics'], name='count').to_frame())
display(pd.crosstab(
    pd.Series([row['error_type'] for row in records], name='error_type'),
    pd.Series([row['source_dataset'] for row in records], name='source_dataset'),
))
print('Unique source IDs:', manifest['output']['unique_source_ids'])
print('Unique leakage groups:', manifest['output']['unique_leakage_groups'])
print('Maximum variants in one leakage group:', manifest['output']['maximum_variants_in_one_leakage_group'])

,examples
AGREEMENT,1000
CASE_ERROR,1000
VERB_CONJUGATION,1000


,count
duplicate_synthetic_ids,0
duplicate_leakage_label_sentence_records,0


source_dataset,UD_Finnish-FTB,UD_Finnish-TDT
error_type,,
AGREEMENT,339,661
CASE_ERROR,365,635
VERB_CONJUGATION,659,341


Unique source IDs: 2816
Unique leakage groups: 2813
Maximum variants in one leakage group: 2


## Deterministic manual-review sample

The manifest fixes 20 IDs per category (60 total). The compact view shows five per class for presentation; reviewers can use the manifest IDs to inspect all 60. The Phase 4 review found 57 clearly valid examples and 3 context-questionable examples (one colloquial verb source and two agreement sources containing unrelated nonstandard/orthographic material), with zero invalid target transformations. Every target transformation introduced its intended category and retained an unambiguous stored correction. This review does **not** establish that all synthetic examples are equally natural learner utterances.

In [8]:
review_ids = set(manifest['manual_review_sample']['synthetic_ids'])
review = [row for row in records if row['synthetic_id'] in review_ids]
review_table = pd.DataFrame([{
    'error_type': row['error_type'],
    'correct': row['correct_sentence'],
    'incorrect': row['incorrect_sentence'],
    'correction': row['correction'],
    'rule': row['generation_rule'],
} for row in review])
assert Counter(review_table['error_type']) == Counter({label: 20 for label in ERROR_TYPES})
review_table.groupby('error_type', sort=True).head(5).reset_index(drop=True)

,error_type,correct,incorrect,correction,rule
0,CASE_ERROR,Komissio on esittänyt teille jo tämän vuoden a...,Komissio on esittänyt teille jo tämän vuoden a...,kokonaisen,adjective_case_mismatch_v1
1,CASE_ERROR,"Siinä ei tuoda esiin sitä johdonmukaisuutta, j...","Siinä ei tuoda esiin sitä johdonmukaisuutta, j...",yhteiseurooppalainen,adjective_case_mismatch_v1
2,CASE_ERROR,"Tutkintonimikkeet kertovat, millaisen ammatill...","Tutkintonimikkeet kertovat, millaista ammatill...",millaisen,adjective_case_mismatch_v1
3,CASE_ERROR,Vanha kone pidetään käynnissä uuden rinnalla .,Vanhan kone pidetään käynnissä uuden rinnalla .,Vanha,adjective_case_mismatch_v1
4,CASE_ERROR,Vanha nainen yski läkähtymäisillään .,Vanhan nainen yski läkähtymäisillään .,Vanha,adjective_case_mismatch_v1
5,VERB_CONJUGATION,"Hän työskentelee sotaveteraanien hyväksi , asi...","Hän työskentelevät sotaveteraanien hyväksi , a...",työskentelee,finite_verb_person_number_mismatch_v1
6,VERB_CONJUGATION,Hän oli perillä viiden tienoilla .,Hän olivat perillä viiden tienoilla .,oli,finite_verb_person_number_mismatch_v1
7,VERB_CONJUGATION,Hän suuttui arvostelun vuoksi .,Hän suutuin arvostelun vuoksi .,suuttui,finite_verb_person_number_mismatch_v1
8,VERB_CONJUGATION,Minä haluan sen kiinni niitten kaikkien naiste...,Minä haluaa sen kiinni niitten kaikkien naiste...,haluan,finite_verb_person_number_mismatch_v1
9,VERB_CONJUGATION,Hän kuului myös Viipurin maalaiskunnan hoitoku...,Hän kuuluivat myös Viipurin maalaiskunnan hoit...,kuului,finite_verb_person_number_mismatch_v1


## Final audit and Phase 5 hand-off

Phase 5 may derive features and perform a group-aware split, but must treat `leakage_group_id` as indivisible so variants of identical NFC-normalized source text cannot cross partitions. It should retain `source_id` for debugging, report per-class distributions after splitting, and evaluate synthetic-data limitations explicitly.

In [9]:
assert set(distribution.index) == set(ERROR_TYPES)
assert distribution.nunique() == 1
assert all(row['source_id'] in sources for row in records)
assert all(row['leakage_group_id'] == sources[row['source_id']]['leakage_group_id'] for row in records)
assert all(manifest['quality_checks'].values())
assert manifest['quality_checks']['no_project_split_created'] is True
{
    'records': len(records),
    'classes': distribution.to_dict(),
    'output_sha256': manifest['output']['sha256'],
    'all_quality_checks_passed': True,
    'project_split_created': False,
}

{'records': 3000,
 'classes': {'AGREEMENT': 1000, 'CASE_ERROR': 1000, 'VERB_CONJUGATION': 1000},
 'output_sha256': '1669cccd1b68eaf04fad5c520ca9d38b5e13a47904d84cda4574f0e623a97fca',
 'all_quality_checks_passed': True,
 'project_split_created': False}